# 🧪 UrbanInsight - دفتر الاختبار / Test Notebook

هذا الدفتر مخصص لاختبار كل مكوّن بشكل مستقل مع عرض البيانات الوسيطة وقياس الأداء.

This notebook is dedicated to testing each component independently with intermediate data visualization and performance measurement.

## المحتويات / Contents
1. اختبار `data_loader` - تحميل الصور
2. اختبار `dino_detector` - كشف المباني
3. اختبار `sam_segmenter` - التقسيم
4. اختبار `post_processor` - المعالجة البعدية
5. اختبار `building_id_generator` - توليد المعرفات
6. اختبار التكامل الكامل - Full Pipeline Test


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# إعداد بيئة الاختبار / Test Environment Setup
# ═══════════════════════════════════════════════════════════════════════

import sys, os, time, json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import yaml

# ── إعداد المسارات ────────────────────────────────────────────────────
PROJECT_ROOT = Path('/content/mash')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ── تحميل الإعدادات ───────────────────────────────────────────────────
def load_cfg(name):
    with open(PROJECT_ROOT / 'configs' / name, 'r', encoding='utf-8') as f:
        return yaml.safe_load(f)

model_cfg      = load_cfg('model_config.yaml')
processing_cfg = load_cfg('processing_config.yaml')
prompt_cfg     = load_cfg('prompt_config.yaml')

WEIGHTS_DIR = Path(model_cfg['environment']['weights_dir'])
OUTPUT_DIR  = Path('/content/test_output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

# ── أداة قياس الوقت ───────────────────────────────────────────────────
class Timer:
    """قياس وقت التنفيذ / Execution time measurement"""
    def __enter__(self):
        self.start = time.perf_counter()
        return self
    def __exit__(self, *args):
        self.elapsed = time.perf_counter() - self.start
        print(f'   ⏱️  الوقت: {self.elapsed:.2f} ثانية')

print('✅ بيئة الاختبار جاهزة / Test environment ready')
print(f'📁 مجلد الإخراج: {OUTPUT_DIR}')


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 🧪 اختبار 1: ImageLoader - تحميل الصور
# Test 1: ImageLoader - Image Loading
# ═══════════════════════════════════════════════════════════════════════

print('═' * 55)
print('   🧪 اختبار ImageLoader')
print('═' * 55)

from utils.data_loader import ImageLoader
import kagglehub, glob as _glob

# تحميل بيانات اختبار
print('⬇️  تحميل بيانات الاختبار...')
dataset_path = kagglehub.dataset_download('ugorjiir/spacenet-2-paris-buildings')
test_paths = sorted(_glob.glob(f'{dataset_path}/**/*.tif', recursive=True))[:3]
assert len(test_paths) > 0, '❌ لا توجد صور اختبار!'

img_dir = os.path.dirname(test_paths[0])

# ── الاختبار 1أ: تحميل صورة واحدة ──────────────────────────────────
print('\n📌 الاختبار 1أ: تحميل صورة واحدة')
loader = ImageLoader(image_dir=img_dir, normalize=True)
with Timer():
    img = loader.load_image(test_paths[0])

print(f'   ✅ الشكل: {img.shape}')
print(f'   ✅ النوع: {img.dtype}')
print(f'   ✅ النطاق: [{img.min():.3f}, {img.max():.3f}]')
assert img.ndim == 3 and img.shape[2] == 3, '❌ الصورة يجب أن تكون (H,W,3)'
assert 0.0 <= img.min() and img.max() <= 1.0, '❌ القيم يجب أن تكون في [0,1]'

# ── الاختبار 1ب: معلومات الصورة ──────────────────────────────────────
print('\n📌 الاختبار 1ب: معلومات الصورة')
info = loader.get_image_info(test_paths[0])
print(f'   ✅ الأبعاد: {info.get("width")}x{info.get("height")}')
print(f'   ✅ الصيغة: {info.get("format")}')

# ── الاختبار 1ج: معالجة الصور الكبيرة ───────────────────────────────
print('\n📌 الاختبار 1ج: معالجة الصور الكبيرة (max_size=256)')
loader_small = ImageLoader(image_dir=img_dir, max_size=256, normalize=True)
with Timer():
    img_small = loader_small.load_image(test_paths[0])
print(f'   ✅ الشكل بعد التقليص: {img_small.shape}')
assert max(img_small.shape[:2]) <= 256, '❌ التقليص فشل!'

# ── الاختبار 1د: تحميل جميع الصور ───────────────────────────────────
print('\n📌 الاختبار 1د: تحميل جميع الصور بالمجلد')
with Timer():
    all_imgs = list(loader.load_all())
print(f'   ✅ عدد الصور المحملة: {len(all_imgs)}')

# ── عرض الصور المحملة ───────────────────────────────────────────────
n = min(3, len(all_imgs))
fig, axes = plt.subplots(1, n, figsize=(5*n, 4))
if n == 1: axes = [axes]
for ax, (arr, path) in zip(axes, all_imgs[:n]):
    ax.imshow(np.clip(arr, 0, 1))
    ax.set_title(path.name[:20], fontsize=8)
    ax.axis('off')
plt.suptitle('🧪 اختبار ImageLoader', fontsize=12)
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'test_data_loader.png'), dpi=80)
plt.show()

print('\n✅ جميع اختبارات ImageLoader نجحت!')


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 🧪 اختبار 2: DINODetector - كشف المباني
# Test 2: DINODetector - Building Detection
# ═══════════════════════════════════════════════════════════════════════

print('═' * 55)
print('   🧪 اختبار DINODetector')
print('═' * 55)

from utils.dino_detector import DINODetector

# تأكد من توفر الأوزان
gdino_weights = WEIGHTS_DIR / 'gdino.pth'
if not gdino_weights.exists():
    print('⬇️  تحميل أوزان DINO...')
    !wget -q --show-progress \
        https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0-alpha/groundingdino_swint_ogc.pth \
        -O {gdino_weights}

dino_config = '/content/GroundingDINO/groundingdino/config/GroundingDINO_SwinT_OGC.py'

# ── الاختبار 2أ: تهيئة النموذج ──────────────────────────────────────
print('\n📌 الاختبار 2أ: تهيئة النموذج')
with Timer():
    detector = DINODetector(
        weights_path=str(gdino_weights),
        config_path=dino_config,
        box_threshold=0.30,
        text_threshold=0.25,
    )
print('   ✅ النموذج محمّل')

# ── الاختبار 2ب: كشف المباني ─────────────────────────────────────────
print('\n📌 الاختبار 2ب: كشف المباني')
test_img = all_imgs[0][0]  # أول صورة محملة
with Timer():
    det_buildings = detector.detect_buildings(test_img)

print(f'   ✅ عدد المباني: {det_buildings["count"]}')
print(f'   ✅ النص المستخدم: {det_buildings["prompt_used"][:50]}...')
if det_buildings['count'] > 0:
    print(f'   ✅ أعلى ثقة: {det_buildings["scores"].max():.3f}')

# ── الاختبار 2ج: كشف متعدد الفئات ───────────────────────────────────
print('\n📌 الاختبار 2ج: كشف متعدد الفئات')
with Timer():
    all_det = detector.detect_all(test_img)
for cat, d in all_det.items():
    print(f'   ✅ {cat}: {d["count"]} كشف')

# ── الاختبار 2د: تصفية الكشوفات ─────────────────────────────────────
print('\n📌 الاختبار 2د: تصفية بحد ثقة 0.4')
filtered = detector.filter_by_score(det_buildings, min_score=0.40)
print(f'   ✅ قبل التصفية: {det_buildings["count"]} | بعد: {filtered["count"]}')

# ── عرض الكشوفات ────────────────────────────────────────────────────
vis = detector.visualize_detections(test_img, det_buildings)
plt.figure(figsize=(10, 6))
plt.imshow(vis)
plt.title(f'🧪 اختبار DINO - {det_buildings["count"]} مبنى مكتشف')
plt.axis('off')
plt.savefig(str(OUTPUT_DIR / 'test_dino_detector.png'), dpi=100)
plt.show()

print('\n✅ جميع اختبارات DINODetector نجحت!')


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 🧪 اختبار 3: SAMSegmenter - التقسيم
# Test 3: SAMSegmenter - Segmentation
# ═══════════════════════════════════════════════════════════════════════

print('═' * 55)
print('   🧪 اختبار SAMSegmenter')
print('═' * 55)

from utils.sam_segmenter import SAMSegmenter

sam_weights = WEIGHTS_DIR / 'sam_vit_h_4b8939.pth'
if not sam_weights.exists():
    print('⬇️  تحميل أوزان SAM (2.5 GB)...')
    !wget -q --show-progress \
        https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth \
        -O {sam_weights}

# ── الاختبار 3أ: تهيئة النموذج ──────────────────────────────────────
print('\n📌 الاختبار 3أ: تهيئة النموذج')
with Timer():
    segmenter = SAMSegmenter(
        checkpoint_path=str(sam_weights),
        model_type='vit_h',
    )
print('   ✅ النموذج محمّل')

# ── الاختبار 3ب: التقسيم من صناديق ─────────────────────────────────
print('\n📌 الاختبار 3ب: التقسيم من صناديق DINO')
if det_buildings['count'] > 0:
    with Timer():
        masks = segmenter.segment_from_boxes(
            image=test_img,
            boxes=det_buildings['boxes'],
            scores=det_buildings['scores'],
        )
    print(f'   ✅ عدد الأقنعة: {len(masks)}')
    if masks:
        areas = [m["area"] for m in masks]
        print(f'   ✅ المساحة المتوسطة: {np.mean(areas):.0f} px²')
        print(f'   ✅ أشكال الأقنعة: {masks[0]["mask"].shape}')
        assert masks[0]['mask'].dtype == bool, '❌ القناع يجب أن يكون نوع bool'
else:
    print('   ⚠️  لا توجد صناديق كشف لاختبار SAM')
    masks = []

# ── الاختبار 3ج: دمج الأقنعة المتداخلة ─────────────────────────────
print(f'\n📌 الاختبار 3ج: دمج الأقنعة المتداخلة')
if len(masks) > 1:
    with Timer():
        merged = segmenter.merge_overlapping_masks(masks, iou_threshold=0.5)
    print(f'   ✅ قبل الدمج: {len(masks)} | بعد: {len(merged)}')
else:
    merged = masks
    print('   ℹ️  لا توجد أقنعة كافية للدمج')

# ── عرض الأقنعة ──────────────────────────────────────────────────────
if masks:
    img_disp = (test_img * 255).astype(np.uint8) if test_img.max() <= 1 else test_img
    overlay = img_disp.copy()
    rng = np.random.default_rng(42)
    colors = rng.integers(50, 255, size=(len(masks), 3))
    for m_idx, m in enumerate(masks):
        overlay[m['mask']] = (overlay[m['mask']] * 0.4 + colors[m_idx] * 0.6).astype(np.uint8)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    ax1.imshow(img_disp); ax1.set_title('الأصلية'); ax1.axis('off')
    ax2.imshow(overlay); ax2.set_title(f'الأقنعة ({len(masks)})'); ax2.axis('off')
    plt.suptitle('🧪 اختبار SAMSegmenter')
    plt.savefig(str(OUTPUT_DIR / 'test_sam_segmenter.png'), dpi=100)
    plt.show()

print('\n✅ جميع اختبارات SAMSegmenter نجحت!')


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 🧪 اختبار 4: PostProcessor - المعالجة البعدية
# Test 4: PostProcessor - Post-Processing
# ═══════════════════════════════════════════════════════════════════════

print('═' * 55)
print('   🧪 اختبار PostProcessor')
print('═' * 55)

from utils.post_processor import PostProcessor

# ── الاختبار 4أ: تهيئة المعالج ──────────────────────────────────────
print('\n📌 الاختبار 4أ: تهيئة المعالج')
processor = PostProcessor(
    min_area=100,
    smoothing_kernel=3,
    closing_iterations=2,
    fill_holes=True,
)
print('   ✅ تم التهيئة')

# ── الاختبار 4ب: معالجة قناع صناعي ──────────────────────────────────
print('\n📌 الاختبار 4ب: معالجة قناع صناعي')
# إنشاء قناع صناعي مع ضوضاء
test_mask = np.zeros((200, 200), dtype=bool)
test_mask[50:150, 50:150] = True   # مبنى رئيسي
test_mask[10:20, 10:20] = True     # ضوضاء صغيرة (يجب حذفها)
test_mask[100:105, 100:105] = False  # ثقب داخلي (يجب ملؤه)

mock_masks = [{'mask': test_mask, 'score': 0.9, 'area': int(test_mask.sum())}]

with Timer():
    cleaned = processor.process(mock_masks)

print(f'   ✅ الأقنعة قبل: {len(mock_masks)} | بعد: {len(cleaned)}')
if cleaned:
    assert 'bbox' in cleaned[0], '❌ يجب أن يكون للقناع bbox'
    assert 'centroid' in cleaned[0], '❌ يجب أن يكون للقناع centroid'
    print(f'   ✅ bbox: {cleaned[0]["bbox"]}')
    print(f'   ✅ centroid: ({cleaned[0]["centroid"][0]:.1f}, {cleaned[0]["centroid"][1]:.1f})')

# ── الاختبار 4ج: الإحصائيات ──────────────────────────────────────────
print('\n📌 الاختبار 4ج: حساب الإحصائيات')
stats = processor.compute_statistics(cleaned)
print(f'   ✅ إجمالي المباني: {stats["total_buildings"]}')
print(f'   ✅ إجمالي المساحة: {stats["total_area_px"]} px²')
print(f'   ✅ المتوسط: {stats["avg_area_px"]:.0f} px²')

# ── الاختبار 4د: خريطة التقسيم ───────────────────────────────────────
print('\n📌 الاختبار 4د: إنشاء خريطة التقسيم')
if masks:  # استخدام الأقنعة من SAM
    cleaned_real = processor.process(masks)
    seg_map = processor.create_segmentation_map(cleaned_real, test_img.shape[:2])
    print(f'   ✅ خريطة التقسيم: {seg_map.shape}')

    img_disp = (test_img * 255).astype(np.uint8) if test_img.max() <= 1 else test_img
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    axes[0].imshow(img_disp); axes[0].set_title('الأصلية'); axes[0].axis('off')
    axes[1].imshow(test_mask, cmap='gray'); axes[1].set_title('قناع اختباري'); axes[1].axis('off')
    axes[2].imshow(seg_map); axes[2].set_title('خريطة التقسيم'); axes[2].axis('off')
    plt.suptitle('🧪 اختبار PostProcessor')
    plt.savefig(str(OUTPUT_DIR / 'test_post_processor.png'), dpi=100)
    plt.show()

print('\n✅ جميع اختبارات PostProcessor نجحت!')


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 🧪 اختبار 5: BuildingIDGenerator - توليد المعرفات
# Test 5: BuildingIDGenerator - ID Generation
# ═══════════════════════════════════════════════════════════════════════

print('═' * 55)
print('   🧪 اختبار BuildingIDGenerator')
print('═' * 55)

from utils.building_id_generator import BuildingIDGenerator

# ── الاختبار 5أ: توليد معرف واحد ────────────────────────────────────
print('\n📌 الاختبار 5أ: توليد معرف واحد')
gen = BuildingIDGenerator(prefix='TEST', project_id='UrbanInsight_Test')

test_mask_gen = np.zeros((100, 100), dtype=bool)
test_mask_gen[20:80, 20:80] = True

with Timer():
    bid1 = gen.generate(
        mask=test_mask_gen,
        score=0.92,
        image_name='test_image.tif'
    )
print(f'   ✅ المعرف: {bid1}')
assert bid1.startswith('TEST-'), '❌ المعرف يجب أن يبدأ بـ TEST-'

# ── الاختبار 5ب: توليد دُفعة ────────────────────────────────────────
print('\n📌 الاختبار 5ب: توليد دُفعة')
mock_masks_batch = [
    {'mask': test_mask_gen, 'score': 0.9, 'area': int(test_mask_gen.sum())},
    {'mask': test_mask_gen, 'score': 0.8, 'area': int(test_mask_gen.sum())},
    {'mask': test_mask_gen, 'score': 0.7, 'area': int(test_mask_gen.sum())},
]
with Timer():
    ids = gen.generate_batch(mock_masks_batch, image_path='/data/test.tif')
print(f'   ✅ عدد المعرفات: {len(ids)}')
print(f'   ✅ عينة: {ids}')
assert len(ids) == 3, '❌ يجب توليد 3 معرفات'
assert len(set(ids)) == 3, '❌ جميع المعرفات يجب أن تكون فريدة'

# ── الاختبار 5ج: البحث عن مبنى ──────────────────────────────────────
print('\n📌 الاختبار 5ج: البحث عن مبنى بالمعرف')
found = gen.get_building_by_id(bid1)
assert found is not None, '❌ المبنى غير موجود!'
print(f'   ✅ المبنى: {found["id"]}, المساحة: {found["geometry"]["area_px"]} px²')

# ── الاختبار 5د: الملخص ──────────────────────────────────────────────
print('\n📌 الاختبار 5د: الملخص الإحصائي')
summary = gen.get_summary()
print(f'   ✅ إجمالي المباني: {summary["total_buildings"]}')
print(f'   ✅ متوسط الثقة: {summary["confidence_stats"]["mean"]:.1%}')

# ── الاختبار 5هـ: حفظ JSON ───────────────────────────────────────────
print('\n📌 الاختبار 5هـ: حفظ JSON')
json_path = str(OUTPUT_DIR / 'test_buildings.json')
with Timer():
    saved = gen.save_to_json(json_path)

# التحقق من الملف
with open(json_path, 'r', encoding='utf-8') as f:
    data = json.load(f)
assert data['metadata']['total_buildings'] == 4, '❌ عدد خاطئ في JSON'
print(f'   ✅ الملف محفوظ: {saved}')
print(f'   ✅ عدد المباني في الملف: {data["metadata"]["total_buildings"]}')

# ── إعادة التعيين ────────────────────────────────────────────────────
gen.reset()
assert len(gen.get_all_buildings()) == 0, '❌ إعادة التعيين فشلت'
print('   ✅ إعادة التعيين تعمل')

print('\n✅ جميع اختبارات BuildingIDGenerator نجحت!')


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 🧪 اختبار 6: اختبار Pipeline الكامل
# Test 6: Full Pipeline Integration Test
# ═══════════════════════════════════════════════════════════════════════

print('═' * 55)
print('   🧪 اختبار Pipeline الكامل')
print('═' * 55)

import time
pipeline_times = {}
test_image_path = str(all_imgs[0][1])  # أول صورة
test_img_full   = all_imgs[0][0]

print(f'\n🖼️  الصورة الاختبارية: {os.path.basename(test_image_path)}')
print(f'   الشكل: {test_img_full.shape}')

# ─── الخطوة 1: تحميل الصورة ──────────────────────────────────────────
t0 = time.perf_counter()
img = loader.load_image(test_image_path)
pipeline_times['loading'] = time.perf_counter() - t0
print(f'\n  1️⃣  تحميل الصورة: ✅ ({pipeline_times["loading"]:.2f}s)')

# ─── الخطوة 2: كشف DINO ───────────────────────────────────────────────
t0 = time.perf_counter()
detections = detector.detect_buildings(img)
detections = detector.filter_by_score(detections, min_score=0.30)
pipeline_times['detection'] = time.perf_counter() - t0
print(f'  2️⃣  كشف DINO: {detections["count"]} مبنى ({pipeline_times["detection"]:.2f}s)')

# ─── الخطوة 3: تقسيم SAM ──────────────────────────────────────────────
if detections['count'] > 0:
    t0 = time.perf_counter()
    raw_masks = segmenter.segment_from_boxes(img, detections['boxes'], detections['scores'])
    pipeline_times['segmentation'] = time.perf_counter() - t0
    print(f'  3️⃣  تقسيم SAM: {len(raw_masks)} قناع ({pipeline_times["segmentation"]:.2f}s)')
else:
    raw_masks = []
    pipeline_times['segmentation'] = 0
    print('  3️⃣  تقسيم SAM: ⚠️ لا كشوفات')

# ─── الخطوة 4: المعالجة البعدية ───────────────────────────────────────
t0 = time.perf_counter()
cleaned_masks = processor.process(raw_masks) if raw_masks else []
pipeline_times['post_processing'] = time.perf_counter() - t0
print(f'  4️⃣  معالجة: {len(cleaned_masks)} مبنى ({pipeline_times["post_processing"]:.2f}s)')

# ─── الخطوة 5: توليد المعرفات ─────────────────────────────────────────
final_gen = BuildingIDGenerator(prefix='BLD', project_id='UrbanInsight')
t0 = time.perf_counter()
building_ids = final_gen.generate_batch(cleaned_masks, image_path=test_image_path)
pipeline_times['id_generation'] = time.perf_counter() - t0
print(f'  5️⃣  توليد المعرفات: {len(building_ids)} ({pipeline_times["id_generation"]:.3f}s)')

# ─── ملخص الأداء ──────────────────────────────────────────────────────
total_time = sum(pipeline_times.values())
print(f'\n⏱️  إجمالي وقت المعالجة: {total_time:.2f} ثانية')
print('\n📊 توزيع الوقت:')
for step, t in pipeline_times.items():
    pct = (t / total_time * 100) if total_time > 0 else 0
    bar = '█' * int(pct / 5)
    print(f'   {step:20s}: {t:.2f}s | {pct:5.1f}% | {bar}')

# ─── عرض النتيجة النهائية ─────────────────────────────────────────────
img_disp = (img * 255).astype(np.uint8) if img.max() <= 1 else img

if cleaned_masks:
    overlay = img_disp.copy().astype(np.float32)
    rng = np.random.default_rng(42)
    colors = rng.integers(50, 255, size=(len(cleaned_masks), 3))
    for m_idx, m in enumerate(cleaned_masks):
        overlay[m['mask']] = overlay[m['mask']] * 0.4 + colors[m_idx] * 0.6
    overlay = overlay.astype(np.uint8)

    fig, axes = plt.subplots(1, 2, figsize=(14, 7))
    axes[0].imshow(img_disp); axes[0].set_title('الصورة الأصلية'); axes[0].axis('off')
    axes[1].imshow(overlay); axes[1].set_title(f'النتيجة النهائية ({len(cleaned_masks)} مبنى)'); axes[1].axis('off')

    # إضافة المعرفات على الصورة
    import matplotlib.patches as mpatches
    for m_idx, (m, bid) in enumerate(zip(cleaned_masks, building_ids)):
        cx, cy = m.get('centroid', (0, 0))
        axes[1].text(cx, cy, bid.split('-')[1], color='white', fontsize=6,
                    ha='center', va='center', fontweight='bold')

    plt.suptitle('🧪 اختبار Pipeline الكامل - UrbanInsight', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(str(OUTPUT_DIR / 'test_full_pipeline.png'), dpi=120, bbox_inches='tight')
    plt.show()

# حفظ نتائج الاختبار
test_results = {
    'image': os.path.basename(test_image_path),
    'detected': detections['count'],
    'segmented': len(raw_masks),
    'cleaned': len(cleaned_masks),
    'ids_generated': len(building_ids),
    'timing_seconds': {k: round(v, 3) for k, v in pipeline_times.items()},
    'total_time_seconds': round(total_time, 3),
}
with open(str(OUTPUT_DIR / 'pipeline_test_results.json'), 'w', encoding='utf-8') as f:
    json.dump(test_results, f, ensure_ascii=False, indent=2)

print(f'\n💾 نتائج الاختبار محفوظة في: {OUTPUT_DIR}/pipeline_test_results.json')
print('\n🎉 اكتمل اختبار Pipeline الكامل!')
